
#  Image Captioning in Malayalam and English using Transformers

 This notebook implements a transformer-based image captioning pipeline. The code:

### 1. Loads required libraries and dataset files.
### 2. Visualizes a few sample images along with their captions.
### 3. Preprocesses images (using torchvision transforms) and text (using a simple tokenizer).
### 4. Defines a CNN encoder + Transformer decoder model for caption generation.
### 5. Provides a training and evaluation loop (with sample evaluation metrics: BLEU, METEOR, ROUGE, CIDEr).
### 6. Runs inference on sample images.
###  7. Saves the final model.

# **Dataset Details**  
### - Captions file (gzipped): `malayalam-visual-genome-test.txt.gz`  
### - Image folders: `train_images`, `test_images`, `dev_images`

### The caption file is expected to be a tab-delimited text file with 7 columns:
### 1. image_id  
### 2. X  
### 3. Y  
### 4. Width  
### 5. Height  
### 6. English Text  
### 7. Malayalam Text

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
!pip install rouge-score
!pip install git+https://github.com/salaniz/pycocoevalcap


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=19491cc93cffcf96efa0af908a3945f4b363a7b0e3c676a8738c4ad7fc3085b4
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score
  Cloning https://github.com/salaniz/pycocoevalcap to /tmp/pip-req-build-jwjwvtcc
  Running command git clone --filter=blob:none --quiet https://github.com/salaniz/pycocoevalcap /tmp/pip-req-build-jwjwvtcc
  Resolved https://github.com/salaniz/pycocoevalcap to commit a24f74c408c918f1f4ec34e9514bc8a76ce41ffd
  Preparing metadata (setup.py) ... done
  Created wheel for pycocoevalcap: filename=pycocoevalcap-1.2-py3-none-any.whl size=104312245 sha256=a4e722f298e26d5200dce817ce5e2886686b447ef366feffec808d5f92a03a13
  Stored in directory: /tmp/pip-ephem-wheel-cache-zs_g_knt/wheels/43/54/73/3e2c6d4ace7657958cde52ac6fd47b342cd4aae5a7aa4fcbf9
Successfully built 

In [3]:
!pip install nltk


In [4]:
!pip install -U nltk


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 25.4 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: nltk
    Found existing installation: nltk 3.2.4
    Uninstalling nltk-3.2.4:
      Successfully uninstalled nltk-3.2.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
preprocessing 0.1.13 requires nltk==3.2.4, but you have nltk 3.9.1 which is incompatible.


# 1 Image Captioning (English)

In [5]:
import os
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.patches as patches

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as models

# Evaluation metric tools setup
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

# BLEU and METEOR are fine with NLTK
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.meteor_score import meteor_score

# ROUGE evaluator from rouge_score
try:
    from rouge_score import rouge_scorer
    rouge_evaluator = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
except ImportError:
    rouge_evaluator = None
    print("rouge_score package not installed; skipping ROUGE metric.")

# CIDEr evaluator
try:
    from pycocoevalcap.cider.cider import Cider
    cider_evaluator = Cider()
except ImportError:
    cider_evaluator = None
    print("pycocoevalcap package not installed; skipping CIDEr metric.")



# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


# import nltk
# nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


In [6]:
#---------------------------------------------
# 1. Load Train and Test Caption Files
#---------------------------------------------
def load_caption_file(caption_file, img_dir, sep='\t'):
    df = pd.read_csv(caption_file, sep=sep, header=None,
                     names=["image_id", "X", "Y", "Width", "Height", "english", "malayalam"])
    # Create a new column for image path
    df["img_path"] = df["image_id"].apply(lambda x: os.path.join(img_dir, f"{x}.jpg"))
    # Filter out rows where the image file does not exist
    df = df[df["img_path"].apply(os.path.exists)]
    return df

TRAIN_CAPTION_FILE = '/kaggle/input/gz-file/malayalam-visual-genome-train.txt'
TEST_CAPTION_FILE  = '/kaggle/input/gz-file/malayalam-visual-genome-test.txt'
TRAIN_IMG_DIR = '/kaggle/input/malayalam-visual-genome-train-images/malayalam-visual-genome-train.images'
TEST_IMG_DIR  = '/kaggle/input/malayalam-visual-genome-test/malayalam-visual-genome-test.images'

train_df = load_caption_file(TRAIN_CAPTION_FILE, TRAIN_IMG_DIR)
test_df  = load_caption_file(TEST_CAPTION_FILE, TEST_IMG_DIR)


In [7]:
#---------------------------------------------
# 2. Visualization Function: Display Samples with Bounding Boxes
#---------------------------------------------
def display_samples(df, image_dir, num_samples=10):
    sample_df = df.sample(num_samples, random_state=42)
    plt.figure(figsize=(15, 15))
    for i, row in enumerate(sample_df.itertuples()):
        # Append '.jpg' extension to match file names
        img_path = os.path.join(image_dir, f"{row.image_id}.jpg")
        if not os.path.exists(img_path):
            print("File not found:", img_path)
            continue
        img = Image.open(img_path).convert('RGB')
        plt.subplot(5, 2, i+1)
        plt.imshow(img)

        # Draw the bounding box using X, Y, Width, Height
        ax = plt.gca()
        rect = patches.Rectangle((row.X, row.Y), row.Width, row.Height,
                                 linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)

        # Show only the English caption to avoid Malayalam font warnings
        plt.title(f"EN: {row.english[:50]}...", fontsize=8)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# Example: Display 10 samples from the test set
display_samples(test_df, TEST_IMG_DIR, num_samples=10)

In [8]:
#---------------------------------------------
# 3. Preprocessing: Image Transformations and Tokenization
#---------------------------------------------
# Image transformations
image_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Simple whitespace tokenizer
def tokenize_caption(caption, vocab, max_len=20):
    tokens = caption.lower().split()
    token_ids = [vocab.get(token, vocab["<unk>"]) for token in tokens]
    if len(token_ids) < max_len:
        token_ids += [vocab["<pad>"]] * (max_len - len(token_ids))
    else:
        token_ids = token_ids[:max_len]
    return token_ids

# Build vocabulary from the training captions (using English text)
def build_vocab(captions, min_freq=1):
    freq = {}
    for caption in captions:
        for token in caption.lower().split():
            freq[token] = freq.get(token, 0) + 1
    # Define special tokens
    vocab = {"<pad>": 0, "<start>": 1, "<end>": 2, "<unk>": 3}
    idx = len(vocab)
    for word, count in freq.items():
        if count >= min_freq:
            vocab[word] = idx
            idx += 1
    return vocab

vocab = build_vocab(train_df["english"].tolist(), min_freq=1)
vocab_size = len(vocab)

In [9]:
#---------------------------------------------
# 4. Custom Dataset Class
#---------------------------------------------
class MVGDataset(Dataset):
    def __init__(self, captions_df, img_dir, transform, vocab, max_len=20, use_english=True):
        self.df = captions_df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.vocab = vocab
        self.max_len = max_len
        self.use_english = use_english

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, f"{row['image_id']}.jpg")
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)

        # Choose caption: English if use_english is True
        caption_text = row["english"] if self.use_english else row["malayalam"]
        # Add start and end tokens
        caption_text = "<start> " + caption_text + " <end>"
        caption_tokens = tokenize_caption(caption_text, self.vocab, self.max_len)
        caption_tensor = torch.tensor(caption_tokens, dtype=torch.long)

        return image, caption_tensor

train_dataset = MVGDataset(train_df, TRAIN_IMG_DIR, image_transform, vocab, max_len=20, use_english=True)
test_dataset  = MVGDataset(test_df, TEST_IMG_DIR, image_transform, vocab, max_len=20, use_english=True)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)


In [10]:
#---------------------------------------------
# 5. Model Definition: Encoder and Transformer Decoder
#---------------------------------------------
class EncoderCNN(nn.Module):
    def __init__(self, encoded_size=256):
        super(EncoderCNN, self).__init__()
        resnet = models.resnet18(pretrained=True)
        modules = list(resnet.children())[:-1]  # remove last fc layer
        self.resnet = nn.Sequential(*modules)
        self.linear = nn.Linear(resnet.fc.in_features, encoded_size)
        self.bn = nn.BatchNorm1d(encoded_size, momentum=0.01)
        for param in self.resnet.parameters():
            param.requires_grad = False

    def forward(self, images):
        with torch.no_grad():
            features = self.resnet(images)
        features = features.view(features.size(0), -1)
        features = self.bn(self.linear(features))
        return features

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:pe[:,1::2].shape[1]])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

class DecoderTransformer(nn.Module):
    def __init__(self, embed_size, vocab_size, num_layers=2, num_heads=4, dropout=0.1, max_len=20):
        super(DecoderTransformer, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.pos_enc = PositionalEncoding(embed_size, max_len=max_len)
        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_size, nhead=num_heads, dropout=dropout)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(embed_size, vocab_size)
        self.embed_size = embed_size

    def forward(self, tgt, memory):
        tgt_embed = self.embed(tgt) * np.sqrt(self.embed_size)
        tgt_embed = self.pos_enc(tgt_embed)
        tgt_embed = tgt_embed.permute(1, 0, 2)
        memory = memory.unsqueeze(0)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_embed.size(0)).to(tgt.device)
        output = self.transformer_decoder(tgt_embed, memory, tgt_mask=tgt_mask)
        output = output.permute(1, 0, 2)
        output = self.fc_out(output)
        return output

class ImageCaptioningModel(nn.Module):
    def __init__(self, encoded_size, embed_size, vocab_size, max_len=20):
        super(ImageCaptioningModel, self).__init__()
        self.encoder = EncoderCNN(encoded_size)
        self.decoder = DecoderTransformer(embed_size, vocab_size, max_len=max_len)

    def forward(self, images, captions):
        features = self.encoder(images)
        outputs = self.decoder(captions[:,:-1], features)
        return outputs

encoded_size = 256
embed_size = 256
max_len = 20

model = ImageCaptioningModel(encoded_size, embed_size, vocab_size, max_len=max_len)
criterion = nn.CrossEntropyLoss(ignore_index=vocab["<pad>"])
optimizer = optim.Adam(model.decoder.parameters(), lr=1e-3)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print("Model ready on device:", device)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 168MB/s]


Model ready on device: cuda


In [11]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [12]:
pip install tqdm


Note: you may need to restart the kernel to use updated packages.


In [13]:
from tqdm import tqdm


In [ ]:
#---------------------------------------------
# 6. Training and Evaluation
#---------------------------------------------
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    epoch_loss = 0
    for images, captions in dataloader:
        images = images.to(device)
        captions = captions.to(device)
        optimizer.zero_grad()
        outputs = model(images, captions)
        targets = captions[:, 1:]  # target tokens: shifted one position
        outputs = outputs.reshape(-1, outputs.size(-1))
        targets = targets.reshape(-1)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(dataloader)

def evaluate_metrics(model, dataloader, device):
    model.eval()
    bleu_scores, meteor_scores = [], []
    rouge_scores = []  # list of dicts for ROUGE-L scores
    cider_scores = []

    # Pre-compute inverse vocabulary mapping
    inv_vocab = {v: k for k, v in vocab.items()}

    with torch.no_grad():
        for images, captions in tqdm(dataloader, desc="Evaluating"):
            images = images.to(device)
            batch_size = images.size(0)
            features = model.encoder(images)
            inputs = torch.full((batch_size, 1), vocab["<start>"], dtype=torch.long).to(device)
            generated = inputs
            # Generate token-by-token for a maximum of max_len tokens
            for _ in range(max_len - 1):
                outputs = model.decoder(generated, features)
                next_token = outputs[:, -1, :].argmax(dim=-1, keepdim=True)
                generated = torch.cat([generated, next_token], dim=1)
            # For each sample in the batch, compute metrics
            for j in range(batch_size):
                ref_tokens = [token for token in captions[j].tolist() if token != vocab["<pad>"]]
                hyp_tokens = generated[j].tolist()
                # BLEU: using token IDs (you could also convert to strings)
                bleu = sentence_bleu([ref_tokens], hyp_tokens, weights=(0.5, 0.5))
                bleu_scores.append(bleu)
                # Convert token IDs to strings
                ref_caption = " ".join([inv_vocab.get(token, "<unk>") for token in ref_tokens])
                hyp_caption = " ".join([inv_vocab.get(token, "<unk>") for token in hyp_tokens])
                # METEOR: requires pre-tokenized lists
                meteor = meteor_score([ref_caption.split()], hyp_caption.split())
                meteor_scores.append(meteor)
                # ROUGE (if evaluator available)
                # ROUGE (if evaluator available)
                if rouge_evaluator is not None:
                    rouge_res = rouge_evaluator.score(ref_caption, hyp_caption)
                    rouge_scores.append(rouge_res['rougeL'].fmeasure)

                # CIDEr (if evaluator available)
                if cider_evaluator is not None:
                    cider, _ = cider_evaluator.compute_score({0: [ref_caption]}, {0: [hyp_caption]})
                    cider_scores.append(cider)

    avg_bleu = np.mean(bleu_scores)
    avg_meteor = np.mean(meteor_scores)
    avg_rouge = np.mean(rouge_scores) if rouge_scores else None
    avg_cider = np.mean(cider_scores) if cider_scores else None
    return avg_bleu, avg_meteor, avg_rouge, avg_cider

# Training loop with progress bars and metric evaluation
num_epochs = 50
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    train_loss = train_one_epoch(model, tqdm(train_loader, desc="Training"), criterion, optimizer, device)
    avg_bleu, avg_meteor, avg_rouge, avg_cider = evaluate_metrics(model, test_loader, device)
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Test BLEU: {avg_bleu:.4f}, METEOR: {avg_meteor:.4f}", end="")
    if avg_rouge is not None:
        print(f", ROUGE-L: {avg_rouge:.4f}", end="")
    if avg_cider is not None:
        print(f", CIDEr: {avg_cider:.4f}", end="")
    print()


Epoch 1/50


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
Evaluating: 100%|██████████| 100/100 [00:13<00:00,  7.22it/s]


Train Loss: 4.0439
Test BLEU: 0.0401, METEOR: 0.2251, ROUGE-L: 0.2231, CIDEr: 0.0000

Epoch 2/50


Evaluating: 100%|██████████| 100/100 [00:10<00:00,  9.95it/s]


Train Loss: 3.5033
Test BLEU: 0.0414, METEOR: 0.2177, ROUGE-L: 0.2202, CIDEr: 0.0000

Epoch 3/50


Training:  85%|████████▌ | 1545/1809 [01:47<00:19, 13.74it/s]

In [ ]:
#---------------------------------------------
# 7. Inference Function
#---------------------------------------------
import matplotlib.pyplot as plt
import torch
from PIL import Image

def generate_caption(model, image_path, device, max_len=20):
    model.eval()
    image = Image.open(image_path).convert('RGB')
    image_tensor = image_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        features = model.encoder(image_tensor)
        inputs = torch.full((1, 1), vocab["<start>"], dtype=torch.long).to(device)
        generated = inputs

        for _ in range(max_len - 1):
            outputs = model.decoder(generated, features)
            next_token = outputs[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
            if next_token.item() == vocab["<end>"]:
                break

    inv_vocab = {v: k for k, v in vocab.items()}
    # Convert token IDs to tokens
    caption_tokens = [inv_vocab.get(token, "<unk>") for token in generated.squeeze().tolist()]
    # Remove the <start> and <end> tokens if they are present
    caption_tokens = [token for token in caption_tokens if token not in ["<start>", "<end>"]]
    caption = " ".join(caption_tokens)
    return caption


# Function to display image with generated caption
def display_image_with_caption(image_path, caption):
    image = Image.open(image_path).convert('RGB')
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.title(caption, fontsize=12)
    plt.show()

# Test inference on a random test image
sample_image = os.path.join(TRAIN_IMG_DIR, f"{train_df.iloc[9]['image_id']}.jpg")
generated_caption = generate_caption(model, sample_image, device)

# Display the image with its generated caption
display_image_with_caption(sample_image, generated_caption)


In [ ]:
# Test inference on a random test image
sample_image = os.path.join(TRAIN_IMG_DIR, f"{train_df.iloc[6]['image_id']}.jpg")
generated_caption = generate_caption(model, sample_image, device)

# Display the image with its generated caption
display_image_with_caption(sample_image, generated_caption)

In [ ]:
# Test inference on a random test image
sample_image = os.path.join(TRAIN_IMG_DIR, f"{train_df.iloc[59]['image_id']}.jpg")
generated_caption = generate_caption(model, sample_image, device)

# Display the image with its generated caption
display_image_with_caption(sample_image, generated_caption)

In [ ]:
# Test inference on a random test image
sample_image = os.path.join(TRAIN_IMG_DIR, f"{train_df.iloc[20]['image_id']}.jpg")
generated_caption = generate_caption(model, sample_image, device)

# Display the image with its generated caption
display_image_with_caption(sample_image, generated_caption)

In [ ]:
#---------------------------------------------
# 8. Save the Trained Model
#---------------------------------------------
MODEL_SAVE_PATH = "image_captioning_model.pth"
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab': vocab,
    'encoded_size': encoded_size,
    'embed_size': embed_size,
    'max_len': max_len
}, MODEL_SAVE_PATH)
print("Model saved to", MODEL_SAVE_PATH)

# Image Captioning (malayalam)

In [ ]:
from tqdm import tqdm

#---------------------------------------------
# 3. Preprocessing: Image Transformations and Tokenization
#---------------------------------------------
# Image transformations
image_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Simple whitespace tokenizer
def tokenize_caption(caption, vocab, max_len=20):
    tokens = caption.lower().split()
    token_ids = [vocab.get(token, vocab["<unk>"]) for token in tokens]
    if len(token_ids) < max_len:
        token_ids += [vocab["<pad>"]] * (max_len - len(token_ids))
    else:
        token_ids = token_ids[:max_len]
    return token_ids

# Build vocabulary from the training captions (using Malayalam text)
def build_vocab(captions, min_freq=1):
    freq = {}
    for caption in captions:
        for token in caption.lower().split():
            freq[token] = freq.get(token, 0) + 1
    # Define special tokens
    vocab = {"<pad>": 0, "<start>": 1, "<end>": 2, "<unk>": 3}
    idx = len(vocab)
    for word, count in freq.items():
        if count >= min_freq:
            vocab[word] = idx
            idx += 1
    return vocab

# Build vocabulary using Malayalam captions
vocab = build_vocab(train_df["malayalam"].tolist(), min_freq=1)
vocab_size = len(vocab)
print("Vocabulary Size:", vocab_size)

#---------------------------------------------
# 4. Custom Dataset Class
#---------------------------------------------
class MVGDataset(Dataset):
    def __init__(self, captions_df, img_dir, transform, vocab, max_len=20):
        self.df = captions_df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, f"{row['image_id']}.jpg")
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)

        # Use Malayalam caption directly
        caption_text = row["malayalam"]
        # Add start and end tokens
        caption_text = "<start> " + caption_text + " <end>"
        caption_tokens = tokenize_caption(caption_text, self.vocab, self.max_len)
        caption_tensor = torch.tensor(caption_tokens, dtype=torch.long)

        return image, caption_tensor



train_dataset = MVGDataset(train_df, TRAIN_IMG_DIR, image_transform, vocab, max_len=20)
test_dataset  = MVGDataset(test_df, TEST_IMG_DIR, image_transform, vocab, max_len=20)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

#---------------------------------------------
# 5. Model Definition: Encoder and Transformer Decoder
#---------------------------------------------
class EncoderCNN(nn.Module):
    def __init__(self, encoded_size=256):
        super(EncoderCNN, self).__init__()
        resnet = models.resnet18(pretrained=True)
        modules = list(resnet.children())[:-1]  # remove last fc layer
        self.resnet = nn.Sequential(*modules)
        self.linear = nn.Linear(resnet.fc.in_features, encoded_size)
        self.bn = nn.BatchNorm1d(encoded_size, momentum=0.01)
        for param in self.resnet.parameters():
            param.requires_grad = False

    def forward(self, images):
        with torch.no_grad():
            features = self.resnet(images)
        features = features.view(features.size(0), -1)
        features = self.bn(self.linear(features))
        return features

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:pe[:,1::2].shape[1]])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

class DecoderTransformer(nn.Module):
    def __init__(self, embed_size, vocab_size, num_layers=2, num_heads=4, dropout=0.1, max_len=20):
        super(DecoderTransformer, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.pos_enc = PositionalEncoding(embed_size, max_len=max_len)
        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_size, nhead=num_heads, dropout=dropout)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(embed_size, vocab_size)
        self.embed_size = embed_size

    def forward(self, tgt, memory):
        tgt_embed = self.embed(tgt) * np.sqrt(self.embed_size)
        tgt_embed = self.pos_enc(tgt_embed)
        tgt_embed = tgt_embed.permute(1, 0, 2)
        memory = memory.unsqueeze(0)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_embed.size(0)).to(tgt.device)
        output = self.transformer_decoder(tgt_embed, memory, tgt_mask=tgt_mask)
        output = output.permute(1, 0, 2)
        output = self.fc_out(output)
        return output

class ImageCaptioningModel(nn.Module):
    def __init__(self, encoded_size, embed_size, vocab_size, max_len=20):
        super(ImageCaptioningModel, self).__init__()
        self.encoder = EncoderCNN(encoded_size)
        self.decoder = DecoderTransformer(embed_size, vocab_size, max_len=max_len)

    def forward(self, images, captions):
        features = self.encoder(images)
        outputs = self.decoder(captions[:,:-1], features)
        return outputs

encoded_size = 256
embed_size = 256
max_len = 20

model = ImageCaptioningModel(encoded_size, embed_size, vocab_size, max_len=max_len)
criterion = nn.CrossEntropyLoss(ignore_index=vocab["<pad>"])
optimizer = optim.Adam(model.decoder.parameters(), lr=1e-3)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print("Model ready on device:", device)




#---------------------------------------------
# 6. Training and Evaluation
#---------------------------------------------
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    epoch_loss = 0
    for images, captions in dataloader:
        images = images.to(device)
        captions = captions.to(device)
        optimizer.zero_grad()
        outputs = model(images, captions)
        targets = captions[:, 1:]  # target tokens: shifted one position
        outputs = outputs.reshape(-1, outputs.size(-1))
        targets = targets.reshape(-1)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(dataloader)

def evaluate_metrics(model, dataloader, device):
    model.eval()
    bleu_scores, meteor_scores = [], []
    rouge_scores = []  # list of dicts for ROUGE-L scores
    cider_scores = []

    # Pre-compute inverse vocabulary mapping
    inv_vocab = {v: k for k, v in vocab.items()}

    with torch.no_grad():
        for images, captions in tqdm(dataloader, desc="Evaluating"):
            images = images.to(device)
            batch_size = images.size(0)
            features = model.encoder(images)
            inputs = torch.full((batch_size, 1), vocab["<start>"], dtype=torch.long).to(device)
            generated = inputs
            # Generate token-by-token for a maximum of max_len tokens
            for _ in range(max_len - 1):
                outputs = model.decoder(generated, features)
                next_token = outputs[:, -1, :].argmax(dim=-1, keepdim=True)
                generated = torch.cat([generated, next_token], dim=1)
            # For each sample in the batch, compute metrics
            for j in range(batch_size):
                ref_tokens = [token for token in captions[j].tolist() if token != vocab["<pad>"]]
                hyp_tokens = generated[j].tolist()
                # BLEU: using token IDs (you could also convert to strings)
                bleu = sentence_bleu([ref_tokens], hyp_tokens, weights=(0.5, 0.5))
                bleu_scores.append(bleu)
                # Convert token IDs to strings
                ref_caption = " ".join([inv_vocab.get(token, "<unk>") for token in ref_tokens])
                hyp_caption = " ".join([inv_vocab.get(token, "<unk>") for token in hyp_tokens])
                # METEOR: requires pre-tokenized lists
                meteor = meteor_score([ref_caption.split()], hyp_caption.split())
                meteor_scores.append(meteor)
                # ROUGE (if evaluator available)
                if 'rouge_evaluator' in globals() and rouge_evaluator is not None:
                    rouge_res = rouge_evaluator.score(ref_caption, hyp_caption)
                    rouge_scores.append(rouge_res['rougeL'].fmeasure)

                # CIDEr (if evaluator available)
                if 'cider_evaluator' in globals() and cider_evaluator is not None:
                    cider, _ = cider_evaluator.compute_score({0: [ref_caption]}, {0: [hyp_caption]})
                    cider_scores.append(cider)

    avg_bleu = np.mean(bleu_scores)
    avg_meteor = np.mean(meteor_scores)
    avg_rouge = np.mean(rouge_scores) if rouge_scores else None
    avg_cider = np.mean(cider_scores) if cider_scores else None
    return avg_bleu, avg_meteor, avg_rouge, avg_cider

# Training loop with progress bars and metric evaluation
num_epochs = 25
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    train_loss = train_one_epoch(model, tqdm(train_loader, desc="Training"), criterion, optimizer, device)
    avg_bleu, avg_meteor, avg_rouge, avg_cider = evaluate_metrics(model, test_loader, device)
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Test BLEU: {avg_bleu:.4f}, METEOR: {avg_meteor:.4f}", end="")
    if avg_rouge is not None:
        print(f", ROUGE-L: {avg_rouge:.4f}", end="")
    if avg_cider is not None:
        print(f", CIDEr: {avg_cider:.4f}", end="")
    print()

In [ ]:
#---------------------------------------------
# 7. Inference Function
#---------------------------------------------
def generate_caption(model, image_path, device, max_len=20):
    model.eval()
    image = Image.open(image_path).convert('RGB')
    image_tensor = image_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        features = model.encoder(image_tensor)
        inputs = torch.full((1, 1), vocab["<start>"], dtype=torch.long).to(device)
        generated = inputs

        for _ in range(max_len - 1):
            outputs = model.decoder(generated, features)
            next_token = outputs[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
            if next_token.item() == vocab["<end>"]:
                break

    inv_vocab = {v: k for k, v in vocab.items()}
    # Convert token IDs to tokens and remove <start> and <end>
    caption_tokens = [inv_vocab.get(token, "<unk>") for token in generated.squeeze().tolist()]
    caption_tokens = [token for token in caption_tokens if token not in ["<start>", "<end>"]]
    caption = " ".join(caption_tokens)
    return caption

def display_image_with_caption(image_path, caption):
    image = Image.open(image_path).convert('RGB')
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis("off")
    # plt.title(caption, fontsize=12)
    plt.show()
    print("\n")
    print("Generated Caption :",caption)

# Test inference on a random test image
sample_image = "/kaggle/input/malayalam-visual-genome-train-images/malayalam-visual-genome-train.images/1005.jpg"
generated_caption = generate_caption(model, sample_image, device)

# Display the image with its generated caption
display_image_with_caption(sample_image, generated_caption)

In [ ]:
# Test inference on a random test image
sample_image =  "/kaggle/input/malayalam-visual-genome-train-images/malayalam-visual-genome-train.images/1159516.jpg"
generated_caption = generate_caption(model, sample_image, device)

# Display the image with its generated caption
display_image_with_caption(sample_image, generated_caption)

In [ ]:
# Test inference on a random test image
sample_image = "/kaggle/input/malayalam-visual-genome-train-images/malayalam-visual-genome-train.images/107957.jpg"
generated_caption = generate_caption(model, sample_image, device)

# Display the image with its generated caption
display_image_with_caption(sample_image, generated_caption)

In [ ]:
# Test inference on a random test image
sample_image = "/kaggle/input/malayalam-visual-genome-train-images/malayalam-visual-genome-train.images/1125.jpg"
generated_caption = generate_caption(model, sample_image, device)

# Display the image with its generated caption
display_image_with_caption(sample_image, generated_caption)

In [ ]:
# Test inference on a random test image
sample_image = "/kaggle/input/malayalam-visual-genome-train-images/malayalam-visual-genome-train.images/1159909.jpg"
generated_caption = generate_caption(model, sample_image, device)

# Display the image with its generated caption
display_image_with_caption(sample_image, generated_caption)